# MOLCOPA 2024 Validation Sample Design

Takes the filtered 2019 points with re-extracted MOLCOPA 2024 class values (output of `filter_samples_by_molca2024.ipynb`) and produces a final validation sample for 2024:

1. Compute Cochran sample size per class.
2. Subsample surplus classes (spatially even across tiles).
3. Generate new random points for deficit classes (spatially spread across unused tiles).
4. Export the final validation dataset as GeoJSON and CSV.

**To re-run for another region**: change `REGION` in Cell 1. The rest of the notebook is region-agnostic.

## Cell 1 — Configuration (only cell to edit between regions)

In [23]:
import os
import platform
from math import ceil
from pyproj import datadir
os.environ["PROJ_LIB"] = datadir.get_data_dir()
from rasterio.crs import CRS
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import transform as warp_transform
from shapely.geometry import Point

REGION = "Africa"  # Options: "Africa", "Amazon_Extension", "Siberia"

if platform.system() == "Windows":
    # Local Windows Environment
    PROJECT_DIR = r"d:\06_Polimi\2025-2026\Land Cover\Validation\sample-design"
else:
    # Leonardo Supercomputer (Linux Environment)
    PROJECT_DIR = "." 

MOLCA_GPKG = os.path.join(PROJECT_DIR, "MOLCA", f"{REGION}_2024_UTM", f"molcx_{REGION.lower()}_2024_s2_exist.gpkg")


# Create output directory dynamically
os.makedirs(os.path.join(PROJECT_DIR, "Samples_2024"), exist_ok=True)

OUTPUT_PREFIX_BY_REGION = {
    "Africa": "africa",
    "Amazon_Extension": "amazonia",
    "Siberia": "siberia",
}

if REGION not in OUTPUT_PREFIX_BY_REGION:
    raise ValueError(f"Unsupported REGION '{REGION}'. Choose one of: {list(OUTPUT_PREFIX_BY_REGION)}")

output_prefix = OUTPUT_PREFIX_BY_REGION[REGION]

INPUT_FILTERED = os.path.join(PROJECT_DIR, "Samples_2024", f"{output_prefix}_static_area_2024_molca_filtered.geojson")
MOLCA_TIFFS    = os.path.join(PROJECT_DIR, "MOLCA", f"{REGION}_2024_UTM", "Tiffs")
LEGEND_CSV     = os.path.join(PROJECT_DIR, "config", "MOLCA_legend.csv")

# Final outputs inherit the same region-specific prefix
OUTPUT_FINAL   = os.path.join(PROJECT_DIR, "Samples_2024", f"{output_prefix}_validation_2024_final.geojson")
OUTPUT_CSV     = os.path.join(PROJECT_DIR, "Samples_2024", f"{output_prefix}_validation_2024_final.csv")


NODATA_CLASS = 255
RANDOM_SEED = 42

# --- Cochran parameters ---
P_EXPECTED  = 0.90   # expected accuracy of MOLCOPA
E_ALLOWABLE = 0.05   # 5% allowable error
Z_ALPHA     = 1.96   # 95% confidence
BUFFER_PCT  = 0.15   # 15% preventive buffer for photo-interpretation discards

rng = np.random.default_rng(RANDOM_SEED)


print("Current OS:     ", platform.system())
print("Region:         ", REGION)
print("Input filtered: ", INPUT_FILTERED)
print("MOLCA gpkg:     ", MOLCA_GPKG)
print("MOLCA tiffs:    ", MOLCA_TIFFS)
print("Legend:         ", LEGEND_CSV)
print("Output GeoJSON: ", OUTPUT_FINAL)
print("Output CSV:     ", OUTPUT_CSV)

Current OS:      Linux
Region:          Africa
Input filtered:  ./Samples_2024/africa_static_area_2024_molca_filtered.geojson
MOLCA gpkg:      ./MOLCA/Africa_2024_UTM/molcx_africa_2024_s2_exist.gpkg
MOLCA tiffs:     ./MOLCA/Africa_2024_UTM/Tiffs
Legend:          ./config/MOLCA_legend.csv
Output GeoJSON:  ./Samples_2024/africa_validation_2024_final.geojson
Output CSV:      ./Samples_2024/africa_validation_2024_final.csv


## Cell 2 — Load inputs

Load the filtered 2019 points, the MOLCA tile index, and the legend. Re-do the spatial join to attach `Tile_ID` to each point (the filter notebook does not persist it). Also build the `tile_classes` lookup describing which classes exist in each tile's raster — needed for generating new points in deficit classes.

In [24]:
# 1) Filtered 2019 points (already have molca_class_2024, molca_label_2024)
kept = gpd.read_file(INPUT_FILTERED)
kept['molca_class_2024'] = kept['molca_class_2024'].astype(int)
print("Filtered points:", kept.shape, " CRS:", kept.crs)
kept.head()

Filtered points: (1192, 7)  CRS: EPSG:4326


,molca_class,molca_label,molca_class_2024,molca_label_2024,class_agrees,Tile_ID,geometry
0,5,Shrubland,7,Grassland,False,35PPS,POINT (28.68598 15.29629)
1,5,Shrubland,20,Forest,False,35PRR,POINT (29.85248 14.30307)
2,5,Shrubland,20,Forest,False,35PQS,POINT (29.15286 14.58446)
3,5,Shrubland,12,Bareland,False,37PES,POINT (39.73347 15.01841)
4,5,Shrubland,12,Bareland,False,37PHP,POINT (42.4963 12.39016)


In [25]:
# 2) MOLCA tile index — collapse to one row per Tile_ID and verify .tif on disk
gdf = gpd.read_file(MOLCA_GPKG)
tiles = gdf[['Tile_ID', 'geometry']].drop_duplicates('Tile_ID').reset_index(drop=True)
tiles['tif_path']   = tiles['Tile_ID'].apply(lambda t: os.path.join(MOLCA_TIFFS, f"MOLCA_{t}_F.tif"))
tiles['tif_exists'] = tiles['tif_path'].apply(os.path.exists)
missing = tiles.loc[~tiles['tif_exists'], 'Tile_ID'].tolist()
if missing:
    print("Tiles listed in gpkg but missing on disk:", missing)
tiles = tiles[tiles['tif_exists']].drop(columns='tif_exists').reset_index(drop=True)
tiles = tiles.to_crs(kept.crs)
print("Tiles with .tif on disk:", len(tiles), " CRS:", tiles.crs)
tiles.head()

Tiles with .tif on disk: 252  CRS: EPSG:4326


,Tile_ID,geometry,tif_path
0,36NZN,"POLYGON ((35.71631 7.22973, 36.70932 7.22273, ...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36NZN_F.tif
1,36NZP,"POLYGON ((35.72206 8.13291, 36.71716 8.12502, ...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36NZP_F.tif
2,36PZA,"POLYGON ((35.78245 14.45638, 36.79951 14.44216...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36PZA_F.tif
3,36PZB,"POLYGON ((35.79408 15.35961, 36.81538 15.34446...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36PZB_F.tif
4,36PZQ,"POLYGON ((35.72851 9.03659, 36.72596 9.02781, ...",./MOLCA/Africa_2024_UTM/Tiffs/MOLCA_36PZQ_F.tif


In [26]:
# 3) Attach Tile_ID to filtered points via spatial join (if not already present)
if 'Tile_ID' not in kept.columns:
    joined = gpd.sjoin(kept, tiles[['Tile_ID', 'geometry']], predicate="within", how="left")
    joined = joined[~joined.index.duplicated(keep='first')]
    if 'index_right' in joined.columns:
        joined = joined.drop(columns='index_right')
    n_no_tile = int(joined['Tile_ID'].isna().sum())
    if n_no_tile:
        print(f"WARNING: {n_no_tile} kept points did not match a tile; dropping them.")
        joined = joined.dropna(subset=['Tile_ID'])
    kept = joined.reset_index(drop=True)
print("Points with Tile_ID:", len(kept), " tiles touched:", kept['Tile_ID'].nunique())

Points with Tile_ID: 1192  tiles touched: 161


In [27]:
# 4) Legend → label map
legend_df = pd.read_csv(LEGEND_CSV)
label_map = dict(zip(legend_df['Class code'], legend_df['Label']))
print(legend_df.to_string(index=False))

 Class code                  Label
          5              Shrubland
          7              Grassland
          8               Cropland
         13               Built-up
         12               Bareland
         16 Permanent ice and snow
         15                  Water
          9                Wetland
         11     Lichens and mosses
         20                 Forest
        255                No data


In [28]:
# 5) tile_classes — which classes exist in each tile's raster (excluding nodata)
tile_classes = {}
for _, row in tiles.iterrows():
    with rasterio.open(row['tif_path']) as src:
        u = np.unique(src.read(1))
    tile_classes[row['Tile_ID']] = set(int(v) for v in u if v != NODATA_CLASS)
print("Scanned tile_classes for", len(tile_classes), "tiles.")

Scanned tile_classes for 252 tiles.


## Cell 3 — Cochran sample size

$$n = \left\lceil \frac{p\,(1-p)}{(E/Z)^2} \right\rceil$$

Apply the preventive buffer for photo-interpretation discards.

In [29]:
n_cochran = ceil((P_EXPECTED * (1 - P_EXPECTED)) / (E_ALLOWABLE / Z_ALPHA) ** 2)
n_target  = n_cochran + ceil(n_cochran * BUFFER_PCT)

all_2024_classes = set().union(*tile_classes.values())
classes_present = sorted(list(all_2024_classes))
num_classes     = len(classes_present)

print(f"Cochran per class:        {n_cochran}")
print(f"With {int(BUFFER_PCT*100)}% buffer:          {n_target}")
print(f"Classes present (2024):   {num_classes}  →  {classes_present}")
print(f"Total target sample size: {n_target * num_classes}")

Cochran per class:        139
With 15% buffer:          160
Classes present (2024):   7  →  [7, 8, 9, 12, 13, 15, 20]
Total target sample size: 1120


## Cell 4 — Surplus vs. deficit per class

In [30]:
rows = []
for code in classes_present:
    n_existing = int((kept['molca_class_2024'] == code).sum())
    if n_existing >= n_target:
        status = 'SURPLUS'
        action = f"subsample to {n_target}"
    else:
        status = 'DEFICIT'
        action = f"keep {n_existing}, add {n_target - n_existing} new"
    rows.append({
        'class_code': code,
        'label':      label_map.get(code, '?'),
        'have':       n_existing,
        'target':     n_target,
        'status':     status,
        'action':     action,
    })

status_df = pd.DataFrame(rows)
print(status_df.to_string(index=False))

 class_code     label  have  target  status               action
          7 Grassland    55     160 DEFICIT keep 55, add 105 new
          8  Cropland   194     160 SURPLUS     subsample to 160
          9   Wetland    34     160 DEFICIT keep 34, add 126 new
         12  Bareland   192     160 SURPLUS     subsample to 160
         13  Built-up    26     160 DEFICIT keep 26, add 134 new
         15     Water   178     160 SURPLUS     subsample to 160
         20    Forest   513     160 SURPLUS     subsample to 160


## Cell 5 — Subsample surplus classes (spatially even)

For each surplus class:
1. Group existing points by tile.
2. First pass: cap each tile at `floor(n_target / num_tiles_with_points)` (random within tile).
3. Second pass: top up from tiles with remaining unselected points, taking 1 extra per tile until the target is met.

Within-tile random selection uses `GeoDataFrame.sample(random_state=...)` for reproducibility.

In [31]:
surplus_selected_list = []
surplus_classes = status_df.loc[status_df['status'] == 'SURPLUS', 'class_code'].tolist()

for code in surplus_classes:
    pts = kept[kept['molca_class_2024'] == code].copy()
    tiles_used = pts['Tile_ID'].unique().tolist()
    num_tiles  = len(tiles_used)
    ideal_cap  = max(1, n_target // num_tiles)

    selected_idx = []
    leftover_per_tile = {}

    # First pass: cap each tile at ideal_cap
    for tid, grp in pts.groupby('Tile_ID'):
        take = min(len(grp), ideal_cap)
        chosen = grp.sample(n=take, random_state=RANDOM_SEED + int(code))
        selected_idx.extend(chosen.index.tolist())
        remaining = grp.drop(index=chosen.index)
        if len(remaining) > 0:
            leftover_per_tile[tid] = remaining

    n_selected  = len(selected_idx)
    n_remaining = n_target - n_selected

    # Second pass: add one extra from each tile (sorted by leftovers desc) until target met
    while n_remaining > 0 and leftover_per_tile:
        ordered = sorted(leftover_per_tile.items(), key=lambda kv: len(kv[1]), reverse=True)
        for tid, remaining in ordered:
            if n_remaining <= 0:
                break
            chosen = remaining.sample(n=1, random_state=RANDOM_SEED + int(code) + len(selected_idx))
            selected_idx.append(chosen.index[0])
            new_rem = remaining.drop(index=chosen.index)
            if len(new_rem) > 0:
                leftover_per_tile[tid] = new_rem
            else:
                del leftover_per_tile[tid]
            n_remaining -= 1

    # Ensure unique indices and cap to n_target (safety in case of multiple runs)
    seen = set()
    uniq_idxs = []
    for idx in selected_idx:
        if idx not in seen:
            seen.add(idx)
            uniq_idxs.append(idx)
    # If we somehow selected more than the target, trim deterministically
    if len(uniq_idxs) > n_target:
        rng_local = np.random.default_rng(RANDOM_SEED + int(code))
        keep = rng_local.choice(uniq_idxs, size=n_target, replace=False).tolist()
        uniq_idxs = keep
    sel = pts.loc[uniq_idxs].copy()
    sel['source'] = 'reused_2019'
    surplus_selected_list.append(sel)
    print(f"  Class {code} ({label_map.get(code, '?')}): selected {len(sel)}/{n_target} across {sel['Tile_ID'].nunique()} tiles (had {len(pts)} in {num_tiles} tiles)")

surplus_selected = (
    pd.concat(surplus_selected_list, ignore_index=True) if surplus_selected_list
    else gpd.GeoDataFrame(columns=list(kept.columns) + ['source'], geometry='geometry', crs=kept.crs)
)
print(f"\nTotal surplus points kept: {len(surplus_selected)}")

  Class 8 (Cropland): selected 160/160 across 36 tiles (had 194 in 36 tiles)
  Class 12 (Bareland): selected 160/160 across 28 tiles (had 192 in 28 tiles)
  Class 15 (Water): selected 160/160 across 77 tiles (had 178 in 77 tiles)
  Class 20 (Forest): selected 160/160 across 81 tiles (had 513 in 81 tiles)

Total surplus points kept: 640


## Cell 6 — Generate new points for deficit classes (spatially spread)

For each deficit class:
1. Keep all existing 2019 points of this class.
2. Find tiles where the class exists in the raster but no existing points fall in them (`tiles_unused`).
3. Order those tiles by centroid (lat, lon) and pick at regular intervals for spatial systematic spread. If `n_new > len(tiles_unused)`, use every unused tile once, then cycle through them adding additional points.
4. From each selected tile, randomly pick one pixel of the target class and convert its UTM coordinate to WGS84.

In [32]:
deficit_existing_list = []
deficit_new_list      = []
deficit_classes = status_df.loc[status_df['status'] == 'DEFICIT', 'class_code'].tolist()

tile_geom_by_id = dict(zip(tiles['Tile_ID'], tiles.geometry))
tile_tif_by_id  = dict(zip(tiles['Tile_ID'], tiles['tif_path']))

for code in deficit_classes:
    existing = kept[kept['molca_class_2024'] == code].copy()
    existing['source'] = 'reused_2019'
    deficit_existing_list.append(existing)
    n_existing = len(existing)
    n_new      = n_target - n_existing

    tiles_with_class    = {t for t, cls in tile_classes.items() if code in cls}
    tiles_with_existing = set(existing['Tile_ID'].unique())
    tiles_unused        = tiles_with_class - tiles_with_existing

    if not tiles_unused:
        # Fall back to any tile with the class
        tiles_unused = tiles_with_class

    unused_gdf = tiles[tiles['Tile_ID'].isin(tiles_unused)].copy()
    # Compute centroids in a projected CRS to avoid geographic-CRS centroid warning
    unused_proj = unused_gdf.to_crs(epsg=3857)
    centroids_proj = unused_proj.geometry.centroid
    centroids_wgs = gpd.GeoSeries(centroids_proj).set_crs(epsg=3857).to_crs(epsg=4326)
    unused_gdf['centroid_x'] = centroids_wgs.x
    unused_gdf['centroid_y'] = centroids_wgs.y
    unused_gdf = unused_gdf.sort_values(['centroid_y', 'centroid_x']).reset_index(drop=True)
    n_unused = len(unused_gdf)

    # Build a list of tile assignments (one entry = one new point to generate)
    if n_new <= n_unused:
        step = n_unused / n_new
        idxs = [int(i * step) for i in range(n_new)]
        tile_assignments = [unused_gdf.iloc[i]['Tile_ID'] for i in idxs]
    else:
        # 1 point per unused tile, then cycle through to add more, evenly
        tile_assignments = unused_gdf['Tile_ID'].tolist()
        extra = n_new - n_unused
        i = 0
        while extra > 0:
            tile_assignments.append(unused_gdf.iloc[i % n_unused]['Tile_ID'])
            i += 1
            extra -= 1

    new_rows = []
    for tid in tile_assignments:
        tif_path = tile_tif_by_id[tid]
        with rasterio.open(tif_path) as src:
            data = src.read(1)
            rs, cs = np.where(data == code)
            if len(rs) == 0:
                continue  # safety: shouldn't happen because tile_classes said the class is here
            j = int(rng.integers(0, len(rs)))
            r, c = int(rs[j]), int(cs[j])
            x, y = src.xy(r, c, offset='center')
            if src.crs != CRS.from_epsg(4326):
                lon, lat = warp_transform(src.crs, CRS.from_epsg(4326), [x], [y])
                lon, lat = float(lon[0]), float(lat[0])
            else:
                lon, lat = float(x), float(y)
        new_rows.append({
            'molca_class_2024': code,
            'molca_label_2024': label_map.get(code, '?'),
            'Tile_ID':          tid,
            'source':           'new_2024',
            'geometry':         Point(lon, lat),
        })

    new_gdf = gpd.GeoDataFrame(new_rows, geometry='geometry', crs='EPSG:4326')
    deficit_new_list.append(new_gdf)
    print(f"  Class {code} ({label_map.get(code, '?')}): kept {n_existing} existing, generated {len(new_gdf)} new from {len(set(tile_assignments))} tiles (had {n_unused} unused candidates)")

deficit_existing = (
    pd.concat(deficit_existing_list, ignore_index=True) if deficit_existing_list
    else gpd.GeoDataFrame(columns=list(kept.columns) + ['source'], geometry='geometry', crs=kept.crs)
)
deficit_new = (
    pd.concat(deficit_new_list, ignore_index=True) if deficit_new_list
    else gpd.GeoDataFrame(columns=['molca_class_2024','molca_label_2024','Tile_ID','source','geometry'], geometry='geometry', crs='EPSG:4326')
)
print(f"\nDeficit existing kept: {len(deficit_existing)}   |   Deficit new generated: {len(deficit_new)}")

  Class 7 (Grassland): kept 55 existing, generated 105 new from 105 tiles (had 222 unused candidates)
  Class 9 (Wetland): kept 34 existing, generated 126 new from 126 tiles (had 158 unused candidates)
  Class 13 (Built-up): kept 26 existing, generated 134 new from 134 tiles (had 234 unused candidates)

Deficit existing kept: 115   |   Deficit new generated: 365


## Cell 7 — Combine and standardize columns

In [33]:
# Make sure each piece has molca_label_2024 populated
for df in (surplus_selected, deficit_existing):
    if 'molca_label_2024' not in df.columns or df['molca_label_2024'].isna().any():
        df['molca_label_2024'] = df['molca_class_2024'].map(label_map)

keep_cols = ['molca_class_2024', 'molca_label_2024', 'Tile_ID', 'source', 'geometry']
parts = []
for df in (surplus_selected, deficit_existing, deficit_new):
    if len(df) == 0:
        continue
    parts.append(df[keep_cols].copy())

final = pd.concat(parts, ignore_index=True)
final = gpd.GeoDataFrame(final, geometry='geometry', crs='EPSG:4326')
final.insert(0, 'sample_id', np.arange(1, len(final) + 1))

print("Final sample shape:", final.shape)
final.head()

Final sample shape: (1120, 6)


,sample_id,molca_class_2024,molca_label_2024,Tile_ID,source,geometry
0,1,8,Cropland,35PRQ,reused_2019,POINT (30.66008 12.68602)
1,2,8,Cropland,36PTS,reused_2019,POINT (31.23983 10.77515)
2,3,8,Cropland,36PUV,reused_2019,POINT (31.81607 13.16353)
3,4,8,Cropland,36PVA,reused_2019,POINT (32.08891 14.45563)
4,5,8,Cropland,36PVA,reused_2019,POINT (32.77085 14.45611)


## Cell 8 — Validation summary

In [34]:
print(f"MOLCOPA 2024 VALIDATION SAMPLE DESIGN — {REGION}")
print("=" * 56)
print(f"Cochran parameters: p={P_EXPECTED}, E={E_ALLOWABLE}, Z={Z_ALPHA}")
print(f"Cochran per class:  {n_cochran}")
print(f"With {int(BUFFER_PCT*100)}% buffer:    {n_target} per class")
print(f"Number of classes:  {num_classes}")
print(f"Total target:       {n_target * num_classes}")
print()

summary_rows = []
for code in classes_present:
    sub = final[final['molca_class_2024'] == code]
    n_reused = int((sub['source'] == 'reused_2019').sum())
    n_new    = int((sub['source'] == 'new_2024').sum())
    summary_rows.append({
        'class':         f"{label_map.get(code,'?')}({code})",
        'reused_2019':   n_reused,
        'new_2024':      n_new,
        'total':         n_reused + n_new,
        'tiles_covered': sub['Tile_ID'].nunique(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.loc['TOTAL'] = {
    'class':         'TOTAL',
    'reused_2019':   summary_df['reused_2019'].sum(),
    'new_2024':      summary_df['new_2024'].sum(),
    'total':         summary_df['total'].sum(),
    'tiles_covered': final['Tile_ID'].nunique(),
}
print(summary_df.to_string(index=False))
print("\nAll points require fresh photo-interpretation for 2024.")

MOLCOPA 2024 VALIDATION SAMPLE DESIGN — Africa
Cochran parameters: p=0.9, E=0.05, Z=1.96
Cochran per class:  139
With 15% buffer:    160 per class
Number of classes:  7
Total target:       1120

       class  reused_2019  new_2024  total  tiles_covered
Grassland(7)           55       105    160            132
 Cropland(8)          160         0    160             36
  Wetland(9)           34       126    160            138
Bareland(12)          160         0    160             28
Built-up(13)           26       134    160            141
   Water(15)          160         0    160             77
  Forest(20)          160         0    160             81
       TOTAL          755       365   1120            237

All points require fresh photo-interpretation for 2024.


## Cell 9 — Export

In [35]:
os.makedirs(os.path.dirname(OUTPUT_FINAL), exist_ok=True)
if os.path.exists(OUTPUT_FINAL):
    os.remove(OUTPUT_FINAL)
final.to_file(OUTPUT_FINAL, driver="GeoJSON")
print("Wrote:", OUTPUT_FINAL)

export = final.copy()
export['longitude'] = export.geometry.x
export['latitude']  = export.geometry.y
export[['sample_id', 'longitude', 'latitude', 'molca_class_2024',
        'molca_label_2024', 'Tile_ID', 'source']].to_csv(OUTPUT_CSV, index=False)
print("Wrote:", OUTPUT_CSV)

Wrote: ./Samples_2024/africa_validation_2024_final.geojson
Wrote: ./Samples_2024/africa_validation_2024_final.csv
